# 08 — Avaliar o detector nas bases externas
Execute após o 07. **Não treina novamente:** usa o checkpoint do notebook 02. CPU funciona; GPU acelera. Comece com 200 imagens de cada classe por base (até 800 imagens). Para avaliar todas, altere `MAX_PER_CLASS=None` e use outro `RUN`.

Inclui teste limpo, JPEG e ruído opcionais, FGSM opcional, métricas por base, matrizes de confusão, ROC, exemplos de acertos/erros e ZIP de resultados. Não faz inferência de raça/gênero. Não escolhe limiar nem hiperparâmetros usando as bases externas. Uma vez usadas para selecionar modelos, deixam de ser um teste externo independente para essa seleção.

O arquivo `detector.pt` precisa existir. Pesos ImageNet aleatórios para esta tarefa não substituem seu detector treinado. Se houver erro na localização, a célula lista checkpoints encontrados, sem escolher automaticamente outro modelo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/tcc_justica')
BASE = ROOT/'dados_externos'
BASE.mkdir(parents=True, exist_ok=True)
print('Pasta persistente:', BASE)

In [ ]:
%pip -q install timm==1.0.22 pandas pillow tqdm matplotlib scikit-learn

## 1. Configuração — comece pelo teste rápido

In [ ]:
import json, hashlib, io, time, shutil, platform
import numpy as np, pandas as pd
import torch, timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
CHECKPOINT=ROOT/'modelos'/'xception_piloto_10'/'detector.pt'
TRAIN_MANIFEST=ROOT/'geracoes'/'piloto_10'/'dataset.csv'
MANIFEST=BASE/'manifesto_externo.csv'
RUN='externos_piloto_200'
MAX_PER_CLASS=200  # None = todas as imagens elegíveis de cada base
ROBUSTEZ=False     # True acrescenta JPEG qualidade 30 e ruído sigma 15/255
FGSM=False         # True acrescenta FGSM não direcionado, epsilon 2/255
SEED=42
THRESHOLD=.5       # fixado antes do teste; não ajuste com os resultados externos
BATCH_SIZE=16 if torch.cuda.is_available() else 4
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=ROOT/'resultados_externos'/RUN
OUT.mkdir(parents=True,exist_ok=True)
assert MANIFEST.is_file(), 'Execute primeiro o notebook 07.'
if not CHECKPOINT.is_file():
    print('Checkpoints encontrados:',*[str(p) for p in (ROOT/'modelos').rglob('*.pt')],sep='\n')
    raise FileNotFoundError(f'Falta {CHECKPOINT}. Ajuste CHECKPOINT para seu detector treinado.')
assert TRAIN_MANIFEST.is_file(), 'Falta o manifesto associado ao treinamento. Ajuste TRAIN_MANIFEST.'
def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
ck=torch.load(CHECKPOINT,map_location='cpu',weights_only=True)
assert ck.get('class_map')=={'real':0,'fake':1}, 'Mapeamento de classes incompatível.'
assert ck.get('arch')=='legacy_xception' and ck.get('size')==299
assert ck.get('mean')==[.5]*3 and ck.get('std')==[.5]*3
assert file_hash(TRAIN_MANIFEST)==ck.get('manifest_sha256'), 'Manifesto diferente daquele registrado no checkpoint.'
print('Dispositivo:',DEVICE,'| Época do checkpoint:',ck.get('epoch'))

## 2. Excluir sobreposição exata e congelar a amostra
Recalculamos os hashes das imagens do manifesto original (incluindo treino, validação e teste) antes de selecionar a amostra externa. Isso detecta cópias exatas, não identidades compartilhadas ou versões recortadas/comprimidas. A ausência de identidade documentada permanece uma limitação.

In [ ]:
def rgb_hash(im):
    im=im.convert('RGB')
    return hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
train=pd.read_csv(TRAIN_MANIFEST)
assert train.image_path.map(lambda s:Path(s).is_file()).all(), 'Imagens do experimento original ausentes; não foi possível conferir sobreposição.'
known=set()
for path in tqdm(train.image_path,desc='Hash do experimento original'):
    with Image.open(path) as im:known.add(rgb_hash(im))
ext=pd.read_csv(MANIFEST)
assert ext.image_id.is_unique and set(ext.label)=={0,1}
overlap=ext[ext.sha256_pixels.isin(known)]
overlap.to_csv(OUT/'sobreposicoes_excluidas.csv',index=False)
ext=ext[~ext.sha256_pixels.isin(known)].sort_values('image_id')
parts=[]
for (base,label),g in ext.groupby(['dataset','label']):
    parts.append(g if MAX_PER_CLASS is None else g.sample(min(MAX_PER_CLASS,len(g)),random_state=SEED))
assert parts, 'Nenhuma imagem externa elegível.'
selected=pd.concat(parts).sort_values(['dataset','label','image_id']).reset_index(drop=True)
for name in ['ciplab','human_faces']:
    assert set(selected.loc[selected.dataset==name,'label'])=={0,1}, f'{name}: falta uma classe.'
conditions=['clean']+(['jpeg30','noise15'] if ROBUSTEZ else [])+(['fgsm2'] if FGSM else [])
config={'checkpoint_sha256':file_hash(CHECKPOINT),'manifest_sha256':file_hash(MANIFEST),
        'train_manifest_sha256':file_hash(TRAIN_MANIFEST),'seed':SEED,'max_per_class':MAX_PER_CLASS,
        'conditions':conditions,'threshold':THRESHOLD,'size':299,'mean':[.5]*3,'std':[.5]*3,
        'timm_version':str(timm.__version__),'torch_version':str(torch.__version__),
        'torchvision_version':__import__('torchvision').__version__,
        'batch_size':BATCH_SIZE,'device':str(DEVICE),
        'selection_sha256':hashlib.sha256(selected.to_csv(index=False).encode()).hexdigest()}
cfg_path=OUT/'config_externo.json'
if cfg_path.exists():
    assert json.loads(cfg_path.read_text())==config, 'Configuração mudou. Escolha outro RUN para preservar os resultados.'
else:cfg_path.write_text(json.dumps(config,indent=2))
selected.to_csv(OUT/'amostra_externa.csv',index=False)
display(selected.groupby(['dataset','label']).size().rename('n').to_frame())
print('Sobreposições exatas excluídas:',len(overlap))

## 3. Preparar modelo e imagens locais

In [ ]:
# Copiar só a amostra para o disco do Colab reduz leituras repetidas do Drive.
LOCAL=Path('/content/tcc_cache_externo')/config['manifest_sha256'][:12]
LOCAL.mkdir(parents=True,exist_ok=True)
for r in tqdm(selected.itertuples(),total=len(selected),desc='Cache local'):
    dst=LOCAL/(r.image_id+Path(r.image_path).suffix.lower())
    if not dst.exists() or dst.stat().st_size!=Path(r.image_path).stat().st_size:
        shutil.copyfile(r.image_path,dst)
    with Image.open(dst) as im:
        assert rgb_hash(im)==r.sha256_pixels, f'Imagem mudou desde o 07: {r.image_path}'
model=timm.create_model(ck['arch'],pretrained=False,num_classes=2)
model.load_state_dict(ck['state_dict'],strict=True)
model=model.to(DEVICE).eval()
for param in model.parameters():param.requires_grad_(False)
resize=transforms.Resize((299,299))
to_tensor=transforms.ToTensor()
class ExternalFaces(Dataset):
    def __init__(self,table,condition):self.table=table.reset_index(drop=True);self.condition=condition
    def __len__(self):return len(self.table)
    def __getitem__(self,i):
        r=self.table.iloc[i]
        with Image.open(LOCAL/(r.image_id+Path(r.image_path).suffix.lower())) as im:
            im=resize(im.convert('RGB'))
            if self.condition=='jpeg30':
                buf=io.BytesIO();im.save(buf,format='JPEG',quality=30);buf.seek(0)
                with Image.open(buf) as decoded:im=decoded.convert('RGB').copy()
            x=to_tensor(im)
        if self.condition=='noise15':
            seed=int(hashlib.sha256((str(SEED)+r.image_id).encode()).hexdigest()[:16],16)%(2**63-1)
            noise=torch.randn(x.shape,generator=torch.Generator().manual_seed(seed))
            x=(x+noise*(15/255)).clamp(0,1)
        return x,int(r.label),i

def predict_batch(x,y,condition):
    x=x.to(DEVICE);y=y.to(DEVICE)
    if condition=='fgsm2':
        x=x.detach().requires_grad_(True)
        loss=torch.nn.functional.cross_entropy(model((x-.5)/.5),y)
        grad=torch.autograd.grad(loss,x)[0]
        x=(x.detach()+(2/255)*grad.sign()).clamp(0,1)
    with torch.no_grad():return model((x-.5)/.5).softmax(1)[:,1].cpu().numpy()

## 4. Medir tempo e executar com retomada por lote
A estimativa usa um lote após aquecimento; é aproximada e não inclui interrupções ou toda a leitura/gravação. Cada lote completo é salvo em arquivo próprio. Ao retomar, execute as células anteriores com a mesma configuração. Se mudar CPU/GPU ou tamanho do lote, use outro RUN.

In [ ]:
sample_loader=DataLoader(ExternalFaces(selected.head(BATCH_SIZE),'clean'),batch_size=BATCH_SIZE,num_workers=0)
x,y,_=next(iter(sample_loader))
_ = predict_batch(x,y,'clean')
t0=time.perf_counter();_ = predict_batch(x,y,'clean');seconds=time.perf_counter()-t0
print(f'Teste limpo: estimativa inicial {seconds/len(x)*len(selected)/60:.1f} minutos para {len(selected)} imagens.')
print('JPEG/ruído acrescentam passagens; FGSM exige também cálculo de gradientes.')
chunks=OUT/'lotes';chunks.mkdir(exist_ok=True)
for condition in conditions:
    ds=ExternalFaces(selected,condition)
    for start in tqdm(range(0,len(ds),BATCH_SIZE),desc=condition):
        stop=min(start+BATCH_SIZE,len(ds));dest=chunks/f'{condition}_{start:07d}.csv'
        if dest.exists():
            old=pd.read_csv(dest)
            assert old.image_id.tolist()==selected.iloc[start:stop].image_id.tolist()
            assert old.condition.eq(condition).all() and old.p_fake.between(0,1).all()
            continue
        batch=[ds[i] for i in range(start,stop)]
        x=torch.stack([b[0] for b in batch]);y=torch.tensor([b[1] for b in batch])
        probs=predict_batch(x,y,condition)
        result=selected.iloc[start:stop].copy()
        result['condition']=condition;result['p_fake']=probs
        result['pred']=(probs>=THRESHOLD).astype(int)
        tmp=dest.with_suffix('.tmp');result.to_csv(tmp,index=False);tmp.replace(dest)
frames=[]
for condition in conditions:
    for start in range(0,len(selected),BATCH_SIZE):frames.append(pd.read_csv(chunks/f'{condition}_{start:07d}.csv'))
pred=pd.concat(frames,ignore_index=True)
assert len(pred)==len(selected)*len(conditions)
assert not pred.duplicated(['image_id','condition']).any()
pred.to_csv(OUT/'predicoes_externas.csv',index=False)
print('Predições concluídas:',len(pred))

## 5. Métricas por base e condição
Classe positiva: falsa/manipulada. FPR = reais classificadas como falsas; FNR = falsas classificadas como reais. ASR é calculada somente para FGSM, entre os exemplos classificados corretamente no teste limpo. Não juntamos as duas bases numa única acurácia. Não calculamos significância ou intervalos por pessoa sem identificadores de identidade.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

def compute_metrics(g):
    y=g.label.to_numpy();p=g.pred.to_numpy();tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return {'n':len(g),'n_real':int(tn+fp),'n_fake':int(tp+fn),'accuracy':accuracy_score(y,p),
        'balanced_accuracy':balanced_accuracy_score(y,p),'precision':precision_score(y,p,zero_division=0),
        'recall':recall_score(y,p,zero_division=0),'f1':f1_score(y,p,zero_division=0),
        'roc_auc':roc_auc_score(y,g.p_fake) if len(set(y))==2 else np.nan,
        'fpr':fp/(tn+fp) if tn+fp else np.nan,'fnr':fn/(tp+fn) if tp+fn else np.nan,
        'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}
rows=[]
for (base,condition),g in pred.groupby(['dataset','condition']):
    row={'dataset':base,'condition':condition,**compute_metrics(g),'asr':np.nan,'asr_denominator':np.nan}
    if condition.startswith('fgsm'):
        clean=pred[(pred.dataset==base)&(pred.condition=='clean')][['image_id','label','pred']]
        joined=g.merge(clean,on='image_id',suffixes=('','_clean'),validate='one_to_one')
        correct=joined.pred_clean==joined.label_clean
        row['asr_denominator']=int(correct.sum())
        row['asr']=float((joined.loc[correct,'pred']!=joined.loc[correct,'label']).mean()) if correct.any() else np.nan
    rows.append(row)
metrics=pd.DataFrame(rows);metrics.to_csv(OUT/'metricas_externas.csv',index=False)
display(metrics.round(4))

## 6. Gráficos e exemplos com predições reais do detector

In [ ]:
import matplotlib.pyplot as plt
clean=pred[pred.condition=='clean']
bases=sorted(clean.dataset.unique())
fig,axes=plt.subplots(1,len(bases),figsize=(5*len(bases),4),squeeze=False)
for ax,base in zip(axes[0],bases):
    g=clean[clean.dataset==base];cm=confusion_matrix(g.label,g.pred,labels=[0,1])
    ax.imshow(cm,cmap='Blues')
    for i in range(2):
        for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Real','Falsa'],yticklabels=['Real','Falsa'],xlabel='Predição',ylabel='Rótulo da base',title=base)
plt.tight_layout();plt.savefig(OUT/'matrizes_externas.png',dpi=200);plt.show()
fig,ax=plt.subplots(figsize=(6,5))
for base in bases:
    g=clean[clean.dataset==base];fpr,tpr,_=roc_curve(g.label,g.p_fake)
    ax.plot(fpr,tpr,label=f'{base}: AUC={roc_auc_score(g.label,g.p_fake):.3f}')
ax.plot([0,1],[0,1],'--',color='gray');ax.set(xlabel='FPR',ylabel='TPR',title='ROC — teste externo limpo');ax.legend()
plt.tight_layout();plt.savefig(OUT/'roc_externa.png',dpi=200);plt.show()
fig,ax=plt.subplots(figsize=(8,4))
metrics.pivot(index='dataset',columns='condition',values='accuracy').plot.bar(ax=ax,rot=0)
ax.set(ylim=(0,1),ylabel='Acurácia',xlabel='',title='Desempenho por base e condição')
plt.tight_layout();plt.savefig(OUT/'comparacao_externa.png',dpi=200);plt.show()
# Seleção reproduzível de até um exemplo de cada resultado por base.
fig,axes=plt.subplots(len(bases),4,figsize=(12,4*len(bases)),squeeze=False)
for i,base in enumerate(bases):
    for j,(truth,guess,title) in enumerate([(0,0,'Real correta'),(0,1,'Falso positivo'),(1,1,'Falsa correta'),(1,0,'Falso negativo')]):
        ax=axes[i,j];ax.axis('off');g=clean[(clean.dataset==base)&(clean.label==truth)&(clean.pred==guess)]
        if len(g):
            r=g.sample(1,random_state=SEED).iloc[0]
            with Image.open(r.image_path) as im:ax.imshow(im.convert('RGB'))
            ax.set_title(f'{base}: {title}\np(falsa)={r.p_fake:.3f}',fontsize=10)
        else:ax.set_title(f'{base}: {title}\nNenhum caso na amostra',fontsize=10)
plt.tight_layout();plt.savefig(OUT/'exemplos_predicoes.png',dpi=180);plt.show()

## 7. Exportar os resultados para análise e pôster

In [ ]:
import zipfile
notes = 'Teste externo do checkpoint existente, sem novo treinamento.\nLimiar fixo em 0,5. Pré-processamento igual ao notebook 02: RGB, Resize 299x299, normalização (x-0,5)/0,5.\nResultados separados por base. Amostra limitada por classe conforme config_externo.json; não representa necessariamente a proporção natural da base.\nJPEG é aplicado após Resize; ruído em pixels [0,1], sigma 15/255; FGSM opcional não direcionado, epsilon 2/255, clamp [0,1].\nRótulos de raça/gênero não disponíveis: estes testes não são uma auditoria demográfica.\nDeduplicação por pixels não garante separação por identidade ou ausência de quase duplicatas entre bases.\nDesempenho baixo pode refletir mudança de domínio, manipulação, resolução ou compressão; não demonstra preconceito.\nNão compare diretamente os números com o pôster antigo como ganho/perda: modelo, dados e protocolo são diferentes.\n'
(OUT/'LEIA_RESULTADOS.txt').write_text(notes)
archive=OUT/'resultados_externos_para_enviar.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p.suffix in {'.csv','.json','.png','.txt'}:z.write(p,p.name)
print('Envie este ZIP aqui:',archive)
from google.colab import files
files.download(str(archive))